 # Comprehensive Data Analysis

 This notebook performs Exploratory Data Analysis, Feature Engineering, Anomaly Detection, and Predictive Analytics on the Northwind-style dataset.

 ## 1. Setup and Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# Define path to data
data_path = r"c:\Users\siddh\OneDrive\Desktop\AccioJob\CapStone Project 1\Data\CSV"

# Load Data
customers = pd.read_csv(f"{data_path}/customers.csv")
orders = pd.read_csv(f"{data_path}/orders.csv")
order_details = pd.read_csv(f"{data_path}/order details.csv")
products = pd.read_csv(f"{data_path}/products.csv")
categories = pd.read_csv(f"{data_path}/categories.csv")


ModuleNotFoundError: No module named 'sklearn'

 ## 2. Exploratory Data Analysis (EDA)

 ### Basic EDA: Summary statistics, missing values, data types

In [ ]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Details": order_details,
    "Products": products,
    "Categories": categories
}

for name, df in datasets.items():
    print(f"--- {name} ---")
    print(df.info())
    print("\nMissing Values:\n", df.isnull().sum())
    print("\nSummary Statistics:\n", df.describe(include='all'))
    print("="*50, "\n")


 ### Advanced Analysis: Multi-table joins

In [ ]:
# Merge tables for comprehensive view
# Orders + Order Details
df_merged = orders.merge(order_details, on='orderID', how='inner')
# + Products
df_merged = df_merged.merge(products, on='productID', how='inner')
# + Categories
df_merged = df_merged.merge(categories, on='categoryID', how='inner')
# + Customers
df_merged = df_merged.merge(customers, on='customerID', how='inner')

# Calculate Revenue per line item
df_merged['Revenue'] = df_merged['unitPrice'] * df_merged['quantity'] * (1 - df_merged['discount'])

# Convert dates
orders['orderDate'] = pd.to_datetime(orders['orderDate'])
orders['shippedDate'] = pd.to_datetime(orders['shippedDate'])
df_merged['orderDate'] = pd.to_datetime(df_merged['orderDate'])

# 1. Customer Purchasing Behavior
top_customers = df_merged.groupby('companyName')['Revenue'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_customers.values, y=top_customers.index, palette='viridis')
plt.title('Top 10 Customers by Revenue')
plt.xlabel('Total Revenue')
plt.ylabel('Customer')
plt.show()

# 2. Regional Market Concentration
regional_revenue = df_merged.groupby('country')['Revenue'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=regional_revenue.values, y=regional_revenue.index, palette='magma')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Total Revenue')
plt.ylabel('Country')
plt.show()

# 3. Product Category Performance
category_revenue = df_merged.groupby('categoryName')['Revenue'].sum().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=category_revenue.values, y=category_revenue.index, palette='cubehelix')
plt.title('Revenue by Product Category')
plt.xlabel('Total Revenue')
plt.ylabel('Category')
plt.show()


 ### Temporal Trends

In [ ]:
# Calculate monthly order volumes and revenue growth
df_merged['Month_Year'] = df_merged['orderDate'].dt.to_period('M')
monthly_trends = df_merged.groupby('Month_Year').agg({'orderID': 'nunique', 'Revenue': 'sum'}).reset_index()
monthly_trends['Month_Year'] = monthly_trends['Month_Year'].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.set_xlabel('Month-Year')
ax1.set_ylabel('Revenue', color='tab:blue')
ax1.plot(monthly_trends['Month_Year'], monthly_trends['Revenue'], color='tab:blue', marker='o', label='Revenue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
plt.xticks(rotation=45)

ax2 = ax1.twinx()
ax2.set_ylabel('Order Volume', color='tab:orange')
ax2.plot(monthly_trends['Month_Year'], monthly_trends['orderID'], color='tab:orange', marker='x', linestyle='dashed', label='Order Volume')
ax2.tick_params(axis='y', labelcolor='tab:orange')

plt.title('Monthly Order Volume and Revenue')
fig.tight_layout()
plt.show()


 ## 3. Engineering & Feature Transformation

 ### Segmentation: Customers by total spend and order frequency

In [ ]:
customer_stats = df_merged.groupby('customerID').agg(
    Total_Spend=('Revenue', 'sum'),
    Order_Frequency=('orderID', 'nunique')
).reset_index()

# Segment into quintiles using qcut
customer_stats['Spend_Quintile'] = pd.qcut(customer_stats['Total_Spend'], 5, labels=['Q1(Low)', 'Q2', 'Q3', 'Q4', 'Q5(High)'])
customer_stats['Freq_Quintile'] = pd.qcut(customer_stats['Order_Frequency'].rank(method='first'), 5, labels=['Q1(Low)', 'Q2', 'Q3', 'Q4', 'Q5(High)'])

plt.figure(figsize=(8, 6))
sns.scatterplot(data=customer_stats, x='Order_Frequency', y='Total_Spend', hue='Spend_Quintile', palette='deep')
plt.title('Customer Segmentation: Spend vs Frequency')
plt.show()


 ### Logistics Metrics: 'Shipping Duration'

In [ ]:
# Calculate 'Shipping Duration'
orders['Shipping_Duration'] = (orders['shippedDate'] - orders['orderDate']).dt.days

plt.figure(figsize=(10, 5))
sns.histplot(orders['Shipping_Duration'].dropna(), bins=20, kde=True, color='teal')
plt.title('Distribution of Shipping Duration (Days)')
plt.xlabel('Days')
plt.show()

# Identify bottlenecks (e.g., duration > 14 days)
bottlenecks = orders[orders['Shipping_Duration'] > 14]
print(f"Number of orders with shipping duration > 14 days: {len(bottlenecks)}")


 ## 4. Anomaly Detection: Revenue Collapses and Spikes

In [ ]:
monthly_revenue = monthly_trends[['Month_Year', 'Revenue']].copy()
monthly_revenue['MoM_Growth'] = monthly_revenue['Revenue'].pct_change()

# Identify anomalies (>50% drop or spike)
monthly_revenue['Anomaly'] = 'Normal'
monthly_revenue.loc[monthly_revenue['MoM_Growth'] > 0.50, 'Anomaly'] = 'Spike'
monthly_revenue.loc[monthly_revenue['MoM_Growth'] < -0.50, 'Anomaly'] = 'Collapse'

plt.figure(figsize=(14, 6))
sns.barplot(data=monthly_revenue, x='Month_Year', y='MoM_Growth', hue='Anomaly', palette={'Normal': 'gray', 'Spike': 'green', 'Collapse': 'red'})
plt.axhline(0, color='black', linewidth=1)
plt.axhline(0.5, color='green', linestyle='--', alpha=0.5)
plt.axhline(-0.5, color='red', linestyle='--', alpha=0.5)
plt.title('Month-over-Month Revenue Growth with Anomaly Detection')
plt.xticks(rotation=45)
plt.ylabel('MoM Growth Rate')
plt.show()

print("Detected Anomalies:")
print(monthly_revenue[monthly_revenue['Anomaly'] != 'Normal'])


 ## 5. Predictive Analytics

 ### Customer Churn Prediction

In [ ]:
# Calculate AverageDaysBetweenOrders
order_dates = orders[['customerID', 'orderDate']].sort_values(['customerID', 'orderDate'])
order_dates['PrevOrderDate'] = order_dates.groupby('customerID')['orderDate'].shift(1)
order_dates['DaysBetween'] = (order_dates['orderDate'] - order_dates['PrevOrderDate']).dt.days

avg_days_between = order_dates.groupby('customerID')['DaysBetween'].mean().reset_index()
avg_days_between.rename(columns={'DaysBetween': 'AvgDaysBetweenOrders'}, inplace=True)

# Merge back to customer stats
customer_data = pd.merge(customer_stats, avg_days_between, on='customerID', how='left')

# Impute NaN for customers with only 1 order
customer_data['AvgDaysBetweenOrders'].fillna(customer_data['AvgDaysBetweenOrders'].median(), inplace=True)

# Define Churn: If days since last order > 2 * AvgDaysBetweenOrders (simplified assumption)
last_order_date = orders.groupby('customerID')['orderDate'].max().reset_index()
last_order_date.rename(columns={'orderDate': 'LastOrderDate'}, inplace=True)
customer_data = pd.merge(customer_data, last_order_date, on='customerID', how='left')

max_date = orders['orderDate'].max()
customer_data['DaysSinceLastOrder'] = (max_date - customer_data['LastOrderDate']).dt.days

customer_data['Churn'] = (customer_data['DaysSinceLastOrder'] > (1.5 * customer_data['AvgDaysBetweenOrders'])).astype(int)

# Logistic Regression Model
X = customer_data[['Total_Spend', 'Order_Frequency', 'AvgDaysBetweenOrders']]
y = customer_data['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Churn Prediction - Classification Report:")
print(classification_report(y_test, y_pred))


 ### Demand Forecasting (Simple Time-Series Example)

In [ ]:
# Forecast product demand based on historical QuantitySold using simple rolling average (as placeholder for ARIMA/Prophet)
# Since setting up statsmodels or prophet might require specific local dependencies, using a generic approach:
quantity_trends = df_merged.groupby('orderDate')['quantity'].sum().reset_index()
quantity_trends.set_index('orderDate', inplace=True)
monthly_qty = quantity_trends.resample('M').sum()

# Simple moving average forecast for next 3 months (Quarter)
monthly_qty['SMA_3'] = monthly_qty['quantity'].rolling(window=3).mean()

plt.figure(figsize=(12, 6))
plt.plot(monthly_qty.index, monthly_qty['quantity'], label='Actual Quantity Sold', marker='o')
plt.plot(monthly_qty.index, monthly_qty['SMA_3'], label='3-Month Moving Average (Forecast Basis)', linestyle='--', color='orange')
plt.title('Monthly Product Demand with Moving Average Forecast')
plt.xlabel('Date')
plt.ylabel('Quantity Sold')
plt.legend()
plt.show()

print("Note: For production demand forecasting, consider using `statsmodels.tsa.arima.model.ARIMA` or `prophet.Prophet`.")
